# 🧠 AgriTrust Risk Scoring Model Development
**Objective**: Develop the v4 iteration of the AGRINET risk scoring model using Isolation Forests and behavioral feature engineering.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

plt.style.use('seaborn-v0_8-muted')

## 1. Load Data
We pull from the `processed` data directory generated by the cleaning pipeline.

In [ ]:
df = pd.read_csv('../../data/processed/user_behavior_v2.csv')
print(f"Loaded {len(df)} user behavior records.")
df.head()

## 2. Feature Selection
Focusing on velocity, transaction amounts, and trust score decay.

In [ ]:
features = ['order_frequency', 'avg_transaction_val', 'trust_score', 'dispute_rate', 'trust_volatility']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Train Isolation Forest
Identifying the 1% of users who exhibit non-standard behavioral patterns.

In [ ]:
model = IsolationForest(contamination=0.01, random_state=42)
df['anomaly_score'] = model.fit_predict(X_scaled)

anomalies = df[df['anomaly_score'] == -1]
print(f"Detected {len(anomalies)} potential fraud patterns.")

## 4. Export for Production

In [ ]:
artifact = {
    'model': model,
    'scaler': scaler,
    'features': features,
    'version': 'v4.1.2-alpha'
}
joblib.dump(artifact, '../../../ml_weights/risk_scorer_v4.pkl')